# SemCor Cross-Encoder Evaluation

这个 notebook 复用 `semcor_embeds_explore.ipynb` 的 SemCor 读取方式，并把每个带 `synset_name`/`synset_definition` 的名词标注转换成 cross-encoder 重排任务。

评测思路：先固定一个目标词，收集它在 SemCor 中出现的全部句子样本和全部候选 synset definition。对于每一句带目标词标记的上下文，令 cross-encoder 在这个词的全部候选 definition 上打分，统计 Top-1 Accuracy 与 MRR。

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


In [ ]:
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder

from prepare_semcor import iter_sentence_records, load_semcor_stats


In [ ]:
BASE_DIR = Path("data/semcor")
NUM_SENTENCES = 37176
DEFAULT_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
DEFAULT_RANDOM_SEED = 13


def load_semcor_sentence_sample(base_dir: Path, num_sentences: int | None = None):
    records = iter_sentence_records(base_dir)
    if num_sentences is None:
        return list(records)
    return list(islice(records, num_sentences))


stats = load_semcor_stats(BASE_DIR)
sentence_records = load_semcor_sentence_sample(BASE_DIR, NUM_SENTENCES)

print(f"Loaded {len(sentence_records)} SemCor sentences")
pd.Series(stats)


In [ ]:
def build_token_char_spans(sentence_text: str, tokens: list[str]) -> list[tuple[int, int]]:
    spans = []
    cursor = 0

    for token in tokens:
        while cursor < len(sentence_text) and sentence_text[cursor].isspace():
            cursor += 1

        start_char = sentence_text.find(token, cursor)
        if start_char < 0:
            raise ValueError(
                f"Could not align token {token!r} inside sentence starting from char {cursor}.\n"
                f"text={sentence_text!r}"
            )

        gap_text = sentence_text[cursor:start_char]
        if any(not char.isspace() for char in gap_text):
            raise ValueError(
                f"Unexpected non-space gap {gap_text!r} before token {token!r}.\n"
                f"text={sentence_text!r}"
            )

        end_char = start_char + len(token)
        spans.append((start_char, end_char))
        cursor = end_char

    return spans


def mark_target_in_sentence(
    sentence_record: dict,
    annotation: dict,
    left_marker: str = "[TGT]",
    right_marker: str = "[/TGT]",
) -> str:
    token_spans = build_token_char_spans(sentence_record["text"], sentence_record["tokens"])
    start_char = token_spans[annotation["token_start"]][0]
    end_char = token_spans[annotation["token_end"] - 1][1]
    sentence_text = sentence_record["text"]
    target_text = sentence_text[start_char:end_char]
    return f"{sentence_text[:start_char]}{left_marker} {target_text} {right_marker}{sentence_text[end_char:]}"


def extract_semcor_cross_encoder_examples(
    sentence_records: list[dict],
    target_word: str | None = None,
    max_examples: int | None = None,
    mark_target: bool = True,
) -> list[dict]:
    examples = []
    normalized_target_word = None if target_word is None else target_word.strip().lower()

    for sentence_record in sentence_records:
        for annotation in sentence_record.get("noun_annotations", []):
            synset_name = annotation.get("synset_name")
            synset_definition = annotation.get("synset_definition")
            if not synset_name or not synset_definition:
                continue

            surface_text = " ".join(annotation.get("tokens", []))
            lemma = annotation.get("lemma")
            if normalized_target_word is not None:
                candidate_terms = {
                    surface_text.strip().lower(),
                    (lemma or "").strip().lower(),
                }
                if normalized_target_word not in candidate_terms:
                    continue

            query_text = sentence_record["text"]
            if mark_target:
                query_text = mark_target_in_sentence(sentence_record, annotation)

            examples.append(
                {
                    "sentence_id": sentence_record["sentence_id"],
                    "sentence_text": sentence_record["text"],
                    "query_text": query_text,
                    "surface_text": surface_text,
                    "lemma": lemma,
                    "synset_name": synset_name,
                    "synset_definition": synset_definition,
                }
            )

            if max_examples is not None and len(examples) >= max_examples:
                return examples

    return examples


def build_synset_candidate_bank(examples: list[dict]) -> list[dict]:
    candidate_map = {}
    for example in examples:
        candidate_map.setdefault(
            example["synset_name"],
            {
                "synset_name": example["synset_name"],
                "definition": example["synset_definition"],
            },
        )
    return list(candidate_map.values())


In [ ]:
example_rows = extract_semcor_cross_encoder_examples(
    sentence_records,
    target_word=TARGET_WORD,
    max_examples=5,
)
candidate_bank = build_synset_candidate_bank(
    extract_semcor_cross_encoder_examples(
        sentence_records,
        target_word=TARGET_WORD,
        max_examples=None,
        mark_target=False,
    )
)

print(f"Target word: {TARGET_WORD}")
print(f"Candidate bank size: {len(candidate_bank)} unique synsets")
pd.DataFrame(example_rows)[
    [
        "sentence_id",
        "surface_text",
        "lemma",
        "synset_name",
        "synset_definition",
        "query_text",
    ]
]


In [ ]:
def evaluate_cross_encoder_on_semcor(
    target_word: str,
    model_name: str = DEFAULT_MODEL_NAME,
    sentence_records: list[dict] | None = None,
    candidate_sentence_records: list[dict] | None = None,
    mark_target: bool = True,
    batch_size: int = 32,
):
    if sentence_records is None:
        sentence_records = load_semcor_sentence_sample(BASE_DIR, NUM_SENTENCES)

    if candidate_sentence_records is None:
        candidate_sentence_records = sentence_records

    examples = extract_semcor_cross_encoder_examples(
        sentence_records=sentence_records,
        target_word=target_word,
        max_examples=None,
        mark_target=mark_target,
    )
    if not examples:
        raise ValueError(f"No SemCor examples were found for target_word={target_word!r}.")

    candidate_bank = build_synset_candidate_bank(
        extract_semcor_cross_encoder_examples(
            sentence_records=candidate_sentence_records,
            target_word=target_word,
            max_examples=None,
            mark_target=False,
        )
    )
    if len(candidate_bank) < 2:
        raise ValueError(
            f"Need at least two distinct synsets for target_word={target_word!r} to evaluate disambiguation."
        )

    model = CrossEncoder(model_name)

    top1_correct = 0
    reciprocal_ranks = []
    evaluation_rows = []

    for example in examples:
        candidates = [
            {
                "synset_name": candidate["synset_name"],
                "definition": candidate["definition"],
                "label": int(candidate["synset_name"] == example["synset_name"]),
            }
            for candidate in candidate_bank
        ]

        pairs = [(example["query_text"], candidate["definition"]) for candidate in candidates]
        scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)

        ranked_candidates = sorted(
            [
                {
                    **candidate,
                    "score": float(score),
                }
                for candidate, score in zip(candidates, scores)
            ],
            key=lambda item: item["score"],
            reverse=True,
        )

        gold_rank = next(
            rank for rank, candidate in enumerate(ranked_candidates, start=1) if candidate["label"] == 1
        )
        top_candidate = ranked_candidates[0]

        top1_correct += int(gold_rank == 1)
        reciprocal_ranks.append(1.0 / gold_rank)
        evaluation_rows.append(
            {
                "sentence_id": example["sentence_id"],
                "surface_text": example["surface_text"],
                "lemma": example["lemma"],
                "gold_synset_name": example["synset_name"],
                "gold_definition": example["synset_definition"],
                "gold_rank": gold_rank,
                "is_top1_correct": gold_rank == 1,
                "top_prediction": top_candidate["synset_name"],
                "top_prediction_definition": top_candidate["definition"],
                "top_prediction_score": top_candidate["score"],
                "query_text": example["query_text"],
            }
        )

    results_df = pd.DataFrame(evaluation_rows)
    metrics = pd.Series(
        {
            "target_word": target_word,
            "model_name": model_name,
            "num_examples": len(examples),
            "num_unique_synsets": len(candidate_bank),
            "candidate_synsets_per_example": len(candidate_bank),
            "top1_accuracy": float(top1_correct / len(examples)),
            "mrr": float(np.mean(reciprocal_ranks)),
            "mean_gold_rank": float(results_df["gold_rank"].mean()),
        }
    )
    return metrics, results_df


In [11]:
TARGET_WORD = "house"
metrics, results_df = evaluate_cross_encoder_on_semcor(
    target_word=TARGET_WORD,
    model_name=DEFAULT_MODEL_NAME,
    sentence_records=sentence_records,
    mark_target=False,
)

metrics


target_word                                                     house
model_name                       cross-encoder/ms-marco-MiniLM-L-6-v2
num_examples                                                      163
num_unique_synsets                                                  9
candidate_synsets_per_example                                       9
top1_accuracy                                                0.042945
mrr                                                          0.215291
mean_gold_rank                                               6.276074
dtype: object

In [12]:
results_df.sort_values(["is_top1_correct", "gold_rank"], ascending=[True, False]).head(10)


,sentence_id,surface_text,lemma,gold_synset_name,gold_definition,gold_rank,is_top1_correct,top_prediction,top_prediction_definition,top_prediction_score,query_text
64,brown1/tagfiles/br-k22.xml:36,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,house.n.04,the audience gathered together in a theatre or...,-10.256495,"Mrs. Hewlitt led the birthcontrol league, Mrs...."
72,brown1/tagfiles/br-k25.xml:7,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,person.n.01,a human being,-8.647230,The debris of his other careers was piled ever...
73,brown1/tagfiles/br-k25.xml:10,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,firm.n.01,the members of a business organization that ow...,-10.348547,Bicycle gear-sets he had once used as the basi...
79,brown1/tagfiles/br-k26.xml:3,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,house.n.06,aristocratic family line,-9.333794,There was no room for company in the tiny Wean...
105,brown1/tagfiles/br-r05.xml:45,House,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,house.n.06,aristocratic family line,-9.632084,Readers of the Reader's Digest are familiar wi...
123,brown2/tagfiles/br-g31.xml:20,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,house.n.06,aristocratic family line,-8.986014,In the following year her father undertook to ...
140,brown2/tagfiles/br-n09.xml:170,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,person.n.01,a human being,-8.079819,When he regained consciousness he was in Lord'...
159,brown2/tagfiles/br-n17.xml:105,house,house,house.n.01,a dwelling that serves as living quarters for ...,9,False,person.n.01,a human being,-8.985668,What Joyce wanted me to do was go to Thor's ho...
11,brown1/tagfiles/br-c01.xml:93,house,house,house.n.01,a dwelling that serves as living quarters for ...,8,False,house.n.06,aristocratic family line,-10.301118,He doesn't think that potting them from a deck...
17,brown1/tagfiles/br-j55.xml:6,house,house,house.n.01,a dwelling that serves as living quarters for ...,8,False,person.n.01,a human being,-6.645755,"Instead, he whirled and ran to his house for a..."
